# 10 — SARS-CoV-2 Mpro 共有結合ドッキングチュートリアル
# SARS-CoV-2 Mpro Covalent Docking Tutorial

**ターゲット**: SARS-CoV-2 Main Protease (Mpro / 3CL プロテアーゼ)  
**事例**: N3 inhibitor (Michael acceptor, 6LU7) vs Nirmatrelvir / Paxlovid (Nitrile, 7VH8)  
**触媒残基**: His41 / Cys145 触媒二残基

---

## Mpro と 2 種類の共有結合ウォーヘッド

```
Mpro 活性部位 (His41 / Cys145 触媒二残基)
          His41
           │  プロトン中継
          Cys145─SH   ← 求核性チオール
           │
    ┌──────┴──────────────────┐
    │  Michael acceptor (N3)  │  → S-C-C-C=O  (不可逆付加体)
    │  Nitrile (Nirmatrelvir) │  → S-C(=NH)-  (可逆的チオイミデート)
    └─────────────────────────┘
```

---

## ウォーヘッド種別の比較

| 特徴 | Michael acceptor (N3, 6LU7) | Nitrile (Nirmatrelvir, 7VH8) |
|------|----------------------------|------------------------------|
| 反応種 | ビニルカルボニル (C=C-C=O) | シアノ基 (C≡N) |
| 反応形式 | **不可逆** (1,4-付加) | **可逆** (チオイミデート形成) |
| 選択性 | 高反応性、他 Cys も標的 | 高選択性、穏やかな反応 |
| 承認状況 | 研究ツール化合物 | **FDA EUA (Paxlovid, 2021)** |
| 骨格 | ペプチド型 | ペプチド様 (より薬らしい) |

---

## このノートブックで学べること

1. Mpro ホモ二量体からのモノマー (chain A) 抽出
2. His41/Cys145 触媒二残基の確認方法
3. Michael acceptor vs Nitrile warhead の SMARTS 識別
4. `generate_conformers_multi()` でペプチド様骨格の多配座生成
5. 可逆・不可逆共有結合ドッキングの UniDock2 設定（両方とも Cys145 SG を指定）
6. `draw_interaction_map()` での His41/Cys145 相互作用可視化

> **Note**: ドッキング実行（Section 6）は UniDock2 + GPU が必要です。

In [ ]:
# CONFIG -----------------------------------------------------------------------
DATA_DIR     = "../data/mpro"    # PDB ダウンロード先
RESULTS_DIR  = "../results/mpro" # ドッキング結果出力先
UNIDOCK2_BIN = "unidock2"         # path to UniDock2 binary
# ------------------------------------------------------------------------------

## 1. セットアップ / Setup

In [ ]:
import urllib.request
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, AllChem
from IPython.display import display

# NOTE: docking runners and preparation tools (gridbox, ligand, receptor)
# will be available in mdatools.docking in a future release.
from docking_analysis import DockingResult, get_reader
from docking_analysis.preparation.receptor import load_receptor, prepare_receptor
from docking_analysis.preparation.gridbox import gridbox_from_ligand
from docking_analysis.preparation.ligand import (
    generate_conformers_multi, enumerate_stereoisomers
)
from mdatools.docking.fingerprints.prolif import ProLIFCalculator
from mdatools.docking.visualization.interaction_map import draw_interaction_map
from mdatools.docking.analysis.properties import calculate_properties
from mdatools.docking.analysis.strain import compute_strain_energy
from mdatools.docking.selection.filters import StrainEnergyFilter

RDLogger.DisableLog("rdApp.warning")

data_dir    = Path(DATA_DIR)
results_dir = Path(RESULTS_DIR)
data_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)
print("Setup complete.")

## 2. PDB 構造の取得 / Download PDB Structures

| PDB ID | リガンド | ウォーヘッド | 分解能 | 特徴 |
|--------|---------|------------|--------|-----|
| **6LU7** | N3 (α-ketoamide 型) | Michael acceptor (不可逆) | 2.16 Å | パンデミック初期公開、教育的価値大 |
| **7VH8** | Nirmatrelvir (Paxlovid) | Nitrile (可逆的共有結合) | 1.8 Å | FDA EUA (2021)、高分解能 |

In [ ]:
PDB_IDS = ["6LU7", "7VH8"]

for pdb_id in PDB_IDS:
    dest = data_dir / f"{pdb_id.lower()}.pdb"
    if not dest.exists() or dest.stat().st_size == 0:
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        print(f"Downloading {pdb_id}...", end=" ")
        urllib.request.urlretrieve(url, dest)
        print(f"→ {dest}")
    else:
        print(f"{pdb_id}: already exists ({dest})")

## 3. HETATM 確認・Cys145 検出 / Inspect HETATM and Locate Cys145

### Mpro ホモ二量体について

Mpro は **Chain A + Chain B** のホモ二量体として結晶化されます。  
ドッキングには **chain A のモノマー**を使用します（chain B はドッキングに影響しない）。

> Mpro の活性は二量体化依存ですが、リガンド結合ポケットは各モノマーに独立して存在します。

In [ ]:
def list_hetatm_residues(pdb_path: Path, min_atoms: int = 6) -> list[dict]:
    WATER_CODES = {"HOH", "WAT", "H2O", "DOD", "D2O"}
    seen = {}
    with open(pdb_path) as f:
        for line in f:
            if not line.startswith("HETATM"):
                continue
            res_name = line[17:20].strip()
            chain    = line[21].strip()
            seq_id   = line[22:26].strip()
            key = (chain, res_name, seq_id)
            if res_name not in WATER_CODES:
                seen[key] = seen.get(key, 0) + 1
    return sorted(
        [{"chain": k[0], "residue": k[1], "seqid": k[2], "n_atoms": v}
         for k, v in seen.items() if v >= min_atoms],
        key=lambda x: -x["n_atoms"]
    )


def find_cys_sg(pdb_path: Path, cys_resid: int = 145, chain: str = "A") -> dict | None:
    """Locate Cys145 SG in ATOM records."""
    with open(pdb_path) as f:
        for line in f:
            if not line.startswith("ATOM"):
                continue
            if (line[12:16].strip() == "SG"
                    and line[17:20].strip() == "CYS"
                    and (chain == "*" or line[21].strip() == chain)):
                try:
                    if int(line[22:26].strip()) == cys_resid:
                        return {
                            "resname": "CYS", "resid": cys_resid, "chain": chain, "atom": "SG",
                            "coords": (float(line[30:38]), float(line[38:46]), float(line[46:54])),
                        }
                except ValueError:
                    continue
    return None


# HETATM 確認 (chain A のみ)
for pdb_id in PDB_IDS:
    residues = list_hetatm_residues(data_dir / f"{pdb_id.lower()}.pdb")
    chain_a  = [r for r in residues if r["chain"] == "A"]
    print(f"\n{pdb_id} — Chain A HETATM 残基 (≥6 heavy atoms):")
    for r in chain_a:
        print(f"  {r['residue']:>4}  seqid={r['seqid']:>5}  atoms={r['n_atoms']}")
    if not chain_a:
        print("  (chain A に該当なし — 全チェイン確認:)")
        for r in residues:
            print(f"  Chain {r['chain']}: {r['residue']:>4}  seqid={r['seqid']:>5}  atoms={r['n_atoms']}")

# Cys145 SG の確認
print()
cys_info = {}
for pdb_id in PDB_IDS:
    info = find_cys_sg(data_dir / f"{pdb_id.lower()}.pdb", cys_resid=145, chain="A")
    if info:
        cys_info[pdb_id] = info
        c = info["coords"]
        print(f"{pdb_id}: Cys145 SG found — Chain A, coords = ({c[0]:.2f}, {c[1]:.2f}, {c[2]:.2f})")
    else:
        print(f"{pdb_id}: Cys145 SG NOT found in chain A — check numbering")

### 3.1 リガンドコードの設定

上の出力の chain A 最大 heavy atoms 残基がリガンドコードです。

- **6LU7** (N3): 通常 `"N3X"` または `"PRD"` 系のコード
- **7VH8** (Nirmatrelvir): `"NIR"` または `"PF7"` 系のコード

In [ ]:
# ↓ 上のセルの出力を見て残基コードを設定
LIGAND_CODES = {
    "6LU7": "PJE",   # N3 inhibitor fragment (RCSB chain C, use chain="*")
    "7VH8": "4WI",   # Nirmatrelvir (RCSB code)
}

## 4. リガンド抽出・ウォーヘッドの確認
## Ligand Extraction and Warhead Identification

### 2 種類のウォーヘッド SMARTS

| ウォーヘッド | SMARTS | 例 |
|------------|--------|---|
| Michael acceptor | `[CH2]=[CH]-C(=O)` | アクリルアミド、ビニルスルホン |
| **Nitrile** | `C#N` | Nirmatrelvir の cyanopyridine |

**Nitrile の反応機構:**  
`Cys145-SH + R-C≡N ⇌ R-C(=NH)-S-Cys145` (可逆的チオイミデート)  
→ 細胞毒性が低く、標的選択性が高い（EGFR Cys797 や BTK Cys481 との交差反応が少ない）

In [ ]:
def extract_ligand_mol(pdb_path: Path, res_code: str, chain: str = "A") -> Chem.Mol | None:
    with open(pdb_path) as f:
        lines = f.readlines()
    hetatm = [l for l in lines
               if l.startswith("HETATM") and l[17:20].strip() == res_code
               and (chain == "*" or l[21].strip() == chain)]
    if not hetatm:
        hetatm = [l for l in lines if l.startswith("HETATM") and l[17:20].strip() == res_code]
    if not hetatm:
        print(f"  WARNING: {res_code} not found in {pdb_path.name}")
        return None
    mol = Chem.MolFromPDBBlock("".join(hetatm) + "END\n", removeHs=True, sanitize=True)
    if mol is None:
        print(f"  WARNING: RDKit could not parse {res_code}")
    return mol


# ウォーヘッド SMARTS
MICHAEL_SMARTS  = "[CH2]=[CH]-C(=O)"   # Michael acceptor (不可逆)
NITRILE_SMARTS  = "C#N"                 # Nitrile (可逆)
michael_pattern = Chem.MolFromSmarts(MICHAEL_SMARTS)
nitrile_pattern = Chem.MolFromSmarts(NITRILE_SMARTS)

WARHEAD_LABELS = {
    "6LU7": "Michael acceptor (irreversible)",
    "7VH8": "Nitrile (reversible covalent)",
}

ligand_mols = {}
for pdb_id, res_code in LIGAND_CODES.items():
    mol = extract_ligand_mol(data_dir / f"{pdb_id.lower()}.pdb", res_code, chain="A")
    if mol is None:
        continue
    mol.SetProp("mol_name", res_code)
    mol.SetProp("pdb_id", pdb_id)
    mol.SetProp("pose_rank", "1")
    mol.SetProp("docking_score", "0.0")
    ligand_mols[pdb_id] = mol

    has_ma      = mol.HasSubstructMatch(michael_pattern)
    has_nitrile = mol.HasSubstructMatch(nitrile_pattern)
    print(f"{pdb_id} [{res_code}] {WARHEAD_LABELS[pdb_id]}:")
    print(f"  {mol.GetNumAtoms()} heavy atoms, Michael acceptor={'✓' if has_ma else '(linked in crystal)'}, "
          f"Nitrile={'✓' if has_nitrile else '(linked in crystal)'}")

if ligand_mols:
    mols_list = list(ligand_mols.values())
    legends   = [f"{pid}\n{WARHEAD_LABELS.get(pid, '')}" for pid in ligand_mols]
    display(Draw.MolsToGridImage(mols_list, molsPerRow=2, subImgSize=(400, 300), legends=legends))

### 4.1 反応前構造の生成

結晶構造は共有結合後の形です。ドッキング入力には反応前構造を使います。

**SMILES 取得先:**
- N3 inhibitor: Jin et al., *Science* 2020 (DOI: 10.1126/science.abb3405) Fig.3 / Supplementary
- Nirmatrelvir: ChEMBL ID `CHEMBL4523759` / PubChem CID `155903882`

**Nirmatrelvir のペプチド様骨格への対応:**  
`generate_conformers_multi()` で多様な配座を生成し、ドッキングのサンプリングを改善します。

In [ ]:
# 反応前構造の SMILES (ChEMBL / 原著論文から取得)

# N3 inhibitor — Michael acceptor warhead (α,β-unsaturated carbonyl)
# (Approximate based on Jin et al. 2020 — verify from source)
N3_SMILES = "CC(C)[C@@H](NC(=O)[C@@H]1CCCN1C(=O)[C@@H](NC(=O)c1ccc(C=O)cc1)CC(=O)N)C(=O)C=C"

# Nirmatrelvir / PF-07321332 (ChEMBL4523759)
NIRMATRELVIR_SMILES = "CC1(C2CC2NC(=O)c2cncc(C#N)c2)[C@@H](NC(=O)C(F)(F)F)C1"

pre_reaction_smiles = {
    "6LU7": N3_SMILES,
    "7VH8": NIRMATRELVIR_SMILES,
}
DRUG_NAMES = {"6LU7": "n3", "7VH8": "nirmatrelvir"}

pre_reaction_mols  = {}
input_sdf_paths    = {}

for pdb_id, smi in pre_reaction_smiles.items():
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        print(f"{pdb_id}: SMILES parse failed — update SMILES from the reference above")
        continue

    # ウォーヘッド確認
    has_ma      = mol.HasSubstructMatch(michael_pattern)
    has_nitrile = mol.HasSubstructMatch(nitrile_pattern)
    print(f"{pdb_id}: Michael={'✓' if has_ma else '✗'}, Nitrile={'✓' if has_nitrile else '✗'}")

    # Nirmatrelvir はペプチド様 → 多配座生成
    if pdb_id == "7VH8":
        print(f"  {pdb_id}: ペプチド様骨格 → generate_conformers_multi() で多配座生成")
        try:
            confs = generate_conformers_multi(mol, n_seeds=4)
            mol_3d = confs[0]  # 最初の配座をドッキング入力に使用
            print(f"  生成した配座数: {len(confs)}")
        except Exception as e:
            print(f"  多配座生成失敗 ({e}) → 単一 EmbedMolecule にフォールバック")
            mol_3d = Chem.AddHs(mol)
            AllChem.EmbedMolecule(mol_3d, AllChem.ETKDGv3())
            AllChem.MMFFOptimizeMolecule(mol_3d)
    else:
        mol_3d = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol_3d, AllChem.ETKDGv3())
        AllChem.MMFFOptimizeMolecule(mol_3d)

    mol_3d.SetProp("mol_name", LIGAND_CODES.get(pdb_id, pdb_id))
    pre_reaction_mols[pdb_id] = mol_3d

    sdf_path = data_dir / f"{DRUG_NAMES[pdb_id]}.sdf"
    writer = Chem.SDWriter(str(sdf_path))
    writer.write(mol_3d)
    writer.close()
    input_sdf_paths[pdb_id] = sdf_path
    print(f"  → {sdf_path}")

# 表示
mols_display = [Chem.RemoveHs(m) for m in pre_reaction_mols.values()]
legs = [f"{'N3 (Michael)' if pid == '6LU7' else 'Nirmatrelvir (Nitrile)'}\n(pre-reaction)" for pid in pre_reaction_mols]
display(Draw.MolsToGridImage(mols_display, molsPerRow=2, subImgSize=(400, 300), legends=legs))

## 5. 受容体準備 — モノマー抽出
## Receptor Preparation — Monomer Extraction

Mpro は PDB に **chain A + chain B** のホモ二量体として収録されています。  
ドッキングでは **chain A のみ**を使用します。

In [ ]:
import MDAnalysis as mda


def extract_chain_a_protein(pdb_path: Path, output_path: Path) -> Path:
    """Extract chain A protein atoms from a multi-chain PDB."""
    u = mda.Universe(str(pdb_path))
    # chainID 'A' または segid 'A' を選択
    atoms = u.select_atoms("protein and (chainID A or segid A or chainID PROA)")
    if len(atoms) == 0:
        # フォールバック: chain が設定されていない場合は全タンパク質を使用
        atoms = u.select_atoms("protein")
        print(f"  Warning: chain A selection empty, using all protein ({len(atoms)} atoms)")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    atoms.write(str(output_path))
    return output_path


receptor_paths = {}
receptor_mols  = {}

for pdb_id in PDB_IDS:
    raw_pdb    = data_dir / f"{pdb_id.lower()}.pdb"
    # Step 1: protein_only (全チェイン、溶媒除去)
    all_chain_pdb = data_dir / f"{pdb_id.lower()}_protein.pdb"
    # Step 2: chain A のみ
    out_pdb       = data_dir / f"{pdb_id.lower()}_receptor.pdb"

    if not out_pdb.exists():
        print(f"Preparing {pdb_id} (chain A monomer)...", end=" ")
        # まず protein_only で溶媒・リガンドを除去
        prepare_receptor(raw_pdb, all_chain_pdb, protein_only=True)
        # 次に chain A を抽出
        extract_chain_a_protein(all_chain_pdb, out_pdb)
        print(f"→ {out_pdb}")
    else:
        print(f"{pdb_id} receptor ready: {out_pdb}")

    rec_mol = load_receptor(out_pdb)
    receptor_paths[pdb_id] = out_pdb
    receptor_mols[pdb_id]  = rec_mol

print("\n受容体準備完了 (chain A モノマー)")

## 5.1 グリッドボックスと Cys145/His41 の確認
## Grid Box Setup and Active Site Verification

Mpro 活性部位は相対的に深いポケット（S1–S4 サブサイト）です。  
グリッドボックスが**Cys145 SG と His41** の両方をカバーしているか確認します。

In [ ]:
def find_his41(pdb_path: Path, chain: str = "A") -> dict | None:
    """Find His41 NE2 atom (proton donor/acceptor for catalytic dyad)."""
    with open(pdb_path) as f:
        for line in f:
            if (line.startswith("ATOM")
                    and line[12:16].strip() in ("NE2", "ND1")
                    and line[17:20].strip() == "HIS"
                    and (line[21].strip() == chain or chain == "*")):
                try:
                    if int(line[22:26].strip()) == 41:
                        return {
                            "resname": "HIS", "resid": 41, "chain": chain,
                            "atom": line[12:16].strip(),
                            "coords": (float(line[30:38]), float(line[38:46]), float(line[46:54])),
                        }
                except ValueError:
                    continue
    return None


gridboxes = {}
for pdb_id, mol in ligand_mols.items():
    if mol.GetNumConformers() == 0:
        AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())

    grid = gridbox_from_ligand(mol, padding=5.0)
    gridboxes[pdb_id] = grid

    cx, cy, cz = grid.center
    sx, sy, sz = grid.size

    label = "N3 (Michael acceptor)" if pdb_id == "6LU7" else "Nirmatrelvir (Nitrile)"
    print(f"\n{pdb_id} [{label}]:")
    print(f"  Grid center (Å): ({cx:.2f}, {cy:.2f}, {cz:.2f})")
    print(f"  Grid size   (Å): ({sx:.2f}, {sy:.2f}, {sz:.2f})")

    # Cys145 の確認
    if pdb_id in cys_info:
        x145, y145, z145 = cys_info[pdb_id]["coords"]
        in_box = (abs(x145 - cx) <= sx/2 and abs(y145 - cy) <= sy/2 and abs(z145 - cz) <= sz/2)
        print(f"  Cys145 SG in box: {'✓' if in_box else '⚠'}")

    # His41 の確認
    his41 = find_his41(data_dir / f"{pdb_id.lower()}.pdb", chain="A")
    if his41:
        x41, y41, z41 = his41["coords"]
        in_box_his = (abs(x41 - cx) <= sx/2 and abs(y41 - cy) <= sy/2 and abs(z41 - cz) <= sz/2)
        print(f"  His41 {his41['atom']} in box : {'✓' if in_box_his else '⚠ increase padding to include His41'}")

## 6. UniDock2 共有結合ドッキング設定と実行
## UniDock2 Covalent Docking Setup and Execution

> ⚠️ このセクションは UniDock2 バイナリと GPU が必要です。

**可逆・不可逆どちらも Cys145 SG を指定:**
```python
covalent_residue_atom_info = [["CYS", 145, "SG"]]
```

> UniDock2 は warhead の反応性（可逆/不可逆）を区別しません。  
> 距離拘束付きで Cys145 SG 近傍にウォーヘッドが配置されるようサンプリングします。  
> 可逆/不可逆の違いは**後解析（binding kinetics, covalent bond geometry）**で評価します。

In [ ]:
# NOTE: docking runners and preparation tools (gridbox, ligand, receptor)
# will be available in mdatools.docking in a future release.
from docking_analysis.docking.unidock2 import UniDock2Runner, UniDock2RunConfig

# Mpro Cys145 共有結合ドッキング設定
cov_config = UniDock2RunConfig(
    unidock2_binary=UNIDOCK2_BIN,
    covalent_docking=True,
    covalent_residue_atom_info=[["CYS", 145, "SG"]],  # Mpro Cys145
    exhaustiveness=512,
    num_pose=10,
    task="screen",
)

runner = UniDock2Runner(cov_config)

print("UniDock2RunConfig (Mpro Cys145 covalent) — YAML preview:")
if gridboxes:
    sample_grid = next(iter(gridboxes.values()))
    yaml_str = runner._build_yaml_config(
        receptor=data_dir / "7vh8_receptor.pdb",
        ligand=data_dir / "nirmatrelvir.sdf",
        grid=sample_grid,
        output_sdf=results_dir / "7vh8_nirmatrelvir_out.sdf",
    )
    print(yaml_str)

In [ ]:
import shutil

docking_results = {}

if shutil.which(UNIDOCK2_BIN) is None:
    print(f"⚠ '{UNIDOCK2_BIN}' not found — skipping docking.")
    print("GPU 環境: docker compose -f docker/docker-compose.yml --profile gpu up")
    print("Section 7 uses crystal structure poses instead.")
else:
    for pdb_id in PDB_IDS:
        if pdb_id not in input_sdf_paths or pdb_id not in gridboxes:
            continue
        out_sdf = results_dir / f"{pdb_id.lower()}_{DRUG_NAMES[pdb_id]}_cov_out.sdf"
        print(f"\nRunning covalent docking: {pdb_id} ({WARHEAD_LABELS.get(pdb_id, '')})...")
        try:
            result = runner.run(
                ligand=input_sdf_paths[pdb_id],
                receptor=receptor_paths[pdb_id],
                grid=gridboxes[pdb_id],
                output=out_sdf,
            )
            docking_results[pdb_id] = result
            print(f"  Top score: {result.scores[0]:.2f} kcal/mol, poses: {len(result.poses)}")
        except RuntimeError as e:
            print(f"  Failed: {e}")

## 7. 結合様式の解析（結晶構造ポーズ使用）
## Binding Mode Analysis (Crystal Poses)

In [ ]:
analysis_results = {}
for pdb_id, mol in ligand_mols.items():
    if pdb_id in docking_results:
        analysis_results[pdb_id] = docking_results[pdb_id]
        print(f"{pdb_id}: using docking result")
    else:
        analysis_results[pdb_id] = DockingResult(
            poses=[mol], scores=[0.0],
            source_file=data_dir / f"{pdb_id.lower()}.pdb",
            backend="crystal",
        )
        print(f"{pdb_id}: using crystal structure pose")

### 7.1 ProLIF — His41/Cys145 触媒二残基との相互作用
### ProLIF — Interactions with His41/Cys145 Catalytic Dyad

**Mpro 活性部位の主要認識残基:**

| 残基 | サブサイト | N3 | Nirmatrelvir |
|------|----------|----|--------------|
| **Cys145** | S1 | ✓ (共有) | ✓ (共有) |
| **His41** | 触媒 | ✓ H-bond | ✓ H-bond |
| Glu166 | S1 | ✓ | ✓ |
| His163 | S1 | ✓ | ✓ |
| Phe140 | S2 | ✓ (π-π) | ✓ |
| Met165 | S2 | ✓ (疎水) | ✓ |
| Leu167 | S3/S4 | ✓ | △ |
| Gln192 | S1' | △ | ✓ |

In [ ]:
calculator = ProLIFCalculator()

fp_results = {}
for pdb_id, result in analysis_results.items():
    rec_mol = receptor_mols.get(pdb_id)
    if rec_mol is None:
        print(f"  {pdb_id}: receptor not loaded — skip")
        continue

    print(f"\n{pdb_id} ({WARHEAD_LABELS.get(pdb_id, '')}) — ProLIF fingerprint...")
    fp_df = calculator.calculate(result.poses, rec_mol, show_progress=True)
    fp_results[pdb_id] = fp_df

    active_cols = fp_df.columns[fp_df.any()].tolist()
    print(f"  検出 ({len(active_cols)} 件):")
    for col in active_cols:
        print(f"    {col}")

In [ ]:
# N3 vs Nirmatrelvir の相互作用比較
if len(fp_results) == 2:
    cols_n3  = set(fp_results["6LU7"].columns[fp_results["6LU7"].any()])
    cols_nir = set(fp_results["7VH8"].columns[fp_results["7VH8"].any()])

    print(f"共通 ({len(cols_n3 & cols_nir)} 件):")
    for c in sorted(cols_n3 & cols_nir): print(f"  {c}")

    print(f"\n6LU7 のみ (N3 / Michael acceptor, {len(cols_n3 - cols_nir)} 件):")
    for c in sorted(cols_n3 - cols_nir): print(f"  {c}")

    print(f"\n7VH8 のみ (Nirmatrelvir / Nitrile, {len(cols_nir - cols_n3)} 件):")
    for c in sorted(cols_nir - cols_n3): print(f"  {c}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
pdb_list = ["6LU7", "7VH8"]
titles   = {
    "6LU7": "N3 Inhibitor (6LU7)\nMichael acceptor — irreversible",
    "7VH8": "Nirmatrelvir (7VH8)\nNitrile — reversible covalent (Paxlovid)",
}

for ax, pdb_id in zip(axes, pdb_list):
    if pdb_id not in fp_results:
        ax.set_title(f"{pdb_id}: data not available")
        continue
    fp_df       = fp_results[pdb_id]
    mol         = ligand_mols[pdb_id]
    active_cols = fp_df.columns[fp_df.any()].tolist()
    if active_cols:
        draw_interaction_map(mol=mol, interactions=active_cols,
                             ax=ax, title=f"{titles[pdb_id]}")
    else:
        ax.set_title(f"{pdb_id}: no interactions detected")

plt.tight_layout()
out_fig = results_dir / "mpro_interaction_map_warhead_comparison.png"
fig.savefig(out_fig, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_fig}")

## 8. ポーズ品質評価 — 歪みエネルギー
## Pose Quality Assessment — Strain Energy

Nirmatrelvir のペプチド様骨格は歪みが大きくなりやすいです。  
`StrainEnergyFilter` で非現実的なポーズを除去します。

In [ ]:
strain_rows = []
for pdb_id, result in analysis_results.items():
    for i, mol in enumerate(result.poses):
        mol_h = Chem.AddHs(mol)
        if mol_h.GetNumConformers() == 0:
            AllChem.EmbedMolecule(mol_h, AllChem.ETKDGv3())
        strain = compute_strain_energy(mol_h)
        strain_rows.append({
            "pdb_id"       : pdb_id,
            "ligand"       : LIGAND_CODES.get(pdb_id, pdb_id),
            "warhead_type" : WARHEAD_LABELS.get(pdb_id, ""),
            "pose_rank"    : i + 1,
            "docking_score": result.scores[i] if i < len(result.scores) else None,
            "strain_energy": strain,
        })

strain_df = pd.DataFrame(strain_rows)
display(strain_df.round(3))

# Nirmatrelvir はペプチド様 → 閾値を少し緩めに (12 kcal/mol)
strain_filter = StrainEnergyFilter(threshold=12.0)
mask     = strain_filter.apply(strain_df)
filtered = strain_df[mask]
print(f"\nStrainEnergyFilter (≤12 kcal/mol): {len(strain_df)} → {len(filtered)} poses")

## 9. 物性比較と結合様式サマリー
## Properties and Binding Mode Summary

In [ ]:
rows = []
for pdb_id, mol in ligand_mols.items():
    props = calculate_properties(mol)
    rows.append({
        "compound"     : f"{LIGAND_CODES[pdb_id]} ({pdb_id})",
        "warhead"      : WARHEAD_LABELS.get(pdb_id, ""),
        "reversibility": "irreversible" if pdb_id == "6LU7" else "reversible",
        **props,
    })

props_df = pd.DataFrame(rows).set_index("compound")
display(props_df.round(3))

### 9.1 ウォーヘッド種別サマリー

| 特徴 | N3 (6LU7) | Nirmatrelvir (7VH8) |
|------|-----------|--------------------|
| ウォーヘッド | Michael acceptor | **Nitrile** |
| 反応形式 | **不可逆** (1,4-付加体) | **可逆** (チオイミデート) |
| Cys145 結合 | ✓ | ✓ |
| His41 H 結合 | ✓ | ✓ |
| Glu166 H 結合 | ✓ | ✓ |
| 選択性 | 反応性高め（汎 Cys） | **高選択性**（立体要求で選択的） |
| 臨床承認 | × (研究ツール) | ✓ **FDA EUA (Paxlovid)** |

**Nitrile warhead が Paxlovid に採用された理由:**
- 可逆的 → 活性を調整しやすく、副作用の管理が容易
- Cys145 への選択性が高い（他の Cys を多く持つ酵素との交差反応を抑制）
- Michael acceptor より低反応性 → 血中安定性が高い

**UniDock2 での設定の違い:**  
可逆・不可逆ともに `covalent_residue_atom_info = [["CYS", 145, "SG"]]` で統一設定。  
可逆/不可逆の区別はドッキングソフトウェアでは扱わず、生物物理実験や MD で評価します。

## 10. 次のステップ / Next Steps

このノートブックで示したワークフロー:

```
PDB ダウンロード → ホモ二量体から chain A 抽出
  → Cys145/His41 確認 → warhead SMARTS 識別
  → generate_conformers_multi() (ペプチド様骨格)
  → UniDock2 covalent docking (Cys145 SG)
  → ProLIF His41/Cys145 接触解析
  → 歪みエネルギー評価 → 化合物選択
```

**発展的な解析:**
- `validate_poses_posebusters()` — Nirmatrelvir のペプチド様結合を検証
- `compute_consensus_score()` — 6LU7 と 7VH8 に対する consensus (複数コンフォーメーション対応)
- Mpro 変異体（Omicron 変異: Pro108Ser 等）に対する耐性変異解析
- Nirmatrelvir 耐性変異 (E166V, A173V) を持つ構造でのドッキング

---

## チュートリアルシリーズ全体像

```
06_egfr_kinase.ipynb      — 非共有結合 (Erlotinib/Lapatinib, DFG-in vs out)
07_gpcr_a2a.ipynb         — GPCR 膜タンパク質 (ZM241385/Adenosine, active/inactive)
08_egfr_covalent.ipynb    — 共有結合 Cys797 (Afatinib/Osimertinib, 2nd/3rd gen)
09_kras_g12c_covalent.ipynb — 共有結合 Cys12 SW2P (ARS-853/Sotorasib, oncology)
10_mpro_covalent.ipynb    — 共有結合 Cys145 (N3/Nirmatrelvir, 可逆 vs 不可逆) ← このノートブック
```

**関連 Issue:** #75, #76, #77, #78, #79